# 07 — ARIMA Forecasting (Professor-Style v4)

**Gold Nexus Alpha — clean professor-style forecasting platform**

This notebook rebuilds the ARIMA stage so it matches the professor-style workflow more carefully:

1. Plot the gold time series first.
2. Build trend and seasonality benchmark models.
3. Check stationarity and autocorrelation with ADF, ACF, and PACF.
4. Compare ARIMA-family candidates separately from deterministic trend benchmarks.
5. Add professor-style hybrid models: trend/seasonality + ARIMA residual correction.
6. Evaluate both static multi-step forecasts and one-step rolling forecasts.
7. Export JSON artifacts for the ARIMA page and model-comparison notebook.

Important methodology rule:

- The final selected ARIMA candidate is chosen only from **ARIMA-family models**, not from plain trend benchmarks.
- The validation/test dates stay locked.
- Rolling one-step evaluation is shown because it is the fair comparison style against Naive and Regression.
- Static long-horizon forecast is still exported as a secondary diagnostic because long-horizon ARIMA forecasts can flatten.


In [ ]:
# ======================================================================================
# CELL 1 — REPO SYNC AND CLEAN RESET
# ======================================================================================
# Purpose:
# - Clone the GitHub repository into Colab.
# - Load GITHUB_TOKEN from Colab Secrets.
# - Keep the same clean Colab → GitHub artifact workflow as prior notebooks.
# ======================================================================================

import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
except Exception:
    userdata = None

REPO_OWNER = "rathee000001"
REPO_NAME  = "nyit-gold-intelligence-2026"
BRANCH     = "main"

BASE_DIR = Path("/content")
PROJECT_DIR = BASE_DIR / REPO_NAME

GITHUB_TOKEN = None
if userdata is not None:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

def run_cmd(cmd, cwd=None, allow_fail=False, display_cmd=None):
    """Run a shell command without printing secrets."""
    shown = display_cmd if display_cmd is not None else cmd
    if isinstance(shown, (list, tuple)):
        shown = " ".join(str(x) for x in shown)
    print(f">> {shown}")
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {shown}")
    return p

# Clean reset.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
    print(f"🧹 Removed existing project folder: {PROJECT_DIR}")

public_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
auth_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if GITHUB_TOKEN else public_url

run_cmd(
    ["git", "clone", "--branch", BRANCH, auth_url, str(PROJECT_DIR)],
    display_cmd=["git", "clone", "--branch", BRANCH, f"https://***@github.com/{REPO_OWNER}/{REPO_NAME}.git", str(PROJECT_DIR)]
)

run_cmd(["git", "config", "user.email", "colab-artifact-bot@gold-nexus-alpha.local"], cwd=str(PROJECT_DIR))
run_cmd(["git", "config", "user.name", "Gold Nexus Alpha Colab"], cwd=str(PROJECT_DIR))
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

print("✅ Repo ready:", PROJECT_DIR)
print("✅ Branch:", BRANCH)
print("✅ UTC time:", datetime.now(timezone.utc).isoformat())


In [ ]:
# ======================================================================================
# CELL 2 — DEPENDENCIES
# ======================================================================================
# Purpose:
# - Load ARIMA/time-series tools.
# - Keep this notebook CPU-safe for statsmodels while still working inside Colab Pro.
# ======================================================================================

import sys
import json
import math
import glob
import warnings
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import statsmodels
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    import statsmodels

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa import stattools
from statsmodels.graphics import tsaplots
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 140)

%matplotlib inline

print("✅ Dependencies loaded")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("statsmodels:", statsmodels.__version__)


In [ ]:
# ======================================================================================
# CELL 3 — PATH SETUP, INPUT DETECTION, AND LOCKED TIME WINDOWS
# ======================================================================================
# Purpose:
# - Locate Notebook 03 output: data/aligned/model_ready_univariate.csv
# - Fall back to weekday_clean_matrix.csv or an uploaded Gold_Matrix CSV only if needed.
# - Lock Dataset A and official validation/test dates.
# ======================================================================================

PROJECT_DIR = Path(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
ALIGNED_DIR = DATA_DIR / "aligned"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
MODELS_ARTIFACTS_DIR = ARTIFACTS_DIR / "models"
PAGES_ARTIFACTS_DIR = ARTIFACTS_DIR / "pages"

for folder in [ALIGNED_DIR, MODELS_ARTIFACTS_DIR, PAGES_ARTIFACTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Locked project rules.
OFFICIAL_FORECAST_CUTOFF_DATE = "2026-03-31"

LONG_UNIVARIATE_START = "1968-01-04"
LONG_UNIVARIATE_END   = OFFICIAL_FORECAST_CUTOFF_DATE

TRAIN_START = "1968-01-04"
TRAIN_END   = "2018-12-31"

VALIDATION_START = "2019-01-01"
VALIDATION_END   = "2022-12-30"

TEST_START = "2023-01-02"
TEST_END   = "2026-03-31"

TARGET_COL = "gold_price"

# Candidate ARIMA orders. This is intentionally professor-readable and not too large.
DIRECT_ARIMA_CANDIDATE_ORDERS = [
    (0, 1, 0),   # random walk benchmark
    (1, 1, 0),
    (0, 1, 1),
    (1, 1, 1),
    (2, 1, 0),
    (0, 1, 2),
    (2, 1, 1),
]

# Residual models are ARMA-style because deterministic trend/seasonality is handled first.
RESIDUAL_ARIMA_CANDIDATE_ORDERS = [
    (1, 0, 0),
    (0, 0, 1),
    (1, 0, 1),
    (2, 0, 1),
]

# Rolling mode. refit=False is much faster and keeps Colab runtime reasonable.
# If you want a heavier run later, change ROLLING_REFIT_EVERY from 0 to 20.
ROLLING_REFIT_EVERY = 0

candidate_inputs = [
    ALIGNED_DIR / "model_ready_univariate.csv",
    ALIGNED_DIR / "weekday_clean_matrix.csv",
    PROJECT_DIR / "Gold_Matrix_M3_Daily_2026-04-30.csv",
]

# Colab upload fallback.
candidate_inputs += sorted(Path("/content").glob("*model*ready*univariate*.csv"))
candidate_inputs += sorted(Path("/content").glob("*Gold*Matrix*.csv"))
candidate_inputs += sorted(Path("/content").glob("*gold*matrix*.csv"))

INPUT_PATH = None
for path in candidate_inputs:
    if path.exists():
        INPUT_PATH = path
        break

if INPUT_PATH is None:
    raise FileNotFoundError(
        "Could not find model_ready_univariate.csv, weekday_clean_matrix.csv, or an uploaded Gold_Matrix CSV. "
        "Run Notebook 03 first or upload the current matrix CSV."
    )

print("✅ Input detected:", INPUT_PATH)

raw_df = pd.read_csv(INPUT_PATH)
raw_df.columns = [str(c).strip() for c in raw_df.columns]

if "date" not in raw_df.columns:
    raise ValueError("Input file must contain a 'date' column.")

if TARGET_COL not in raw_df.columns:
    raise ValueError(f"Input file must contain target column '{TARGET_COL}'.")

raw_df["date"] = pd.to_datetime(raw_df["date"], errors="coerce")
raw_df = raw_df.dropna(subset=["date"]).sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)

# If the fallback source is a raw matrix, remove Saturday/Sunday rows here.
# Official wording remains weekday-clean, not full trading-day.
raw_df["weekday"] = raw_df["date"].dt.dayofweek
weekend_rows_detected = int(raw_df["weekday"].isin([5, 6]).sum())
if weekend_rows_detected > 0:
    print(f"⚠️ Weekend rows detected in fallback input: {weekend_rows_detected}. Removing Saturday/Sunday rows.")
    raw_df = raw_df[~raw_df["weekday"].isin([5, 6])].copy()

raw_df = raw_df.drop(columns=["weekday"], errors="ignore")

# Enforce locked long univariate window.
model_df = raw_df[
    (raw_df["date"] >= pd.Timestamp(LONG_UNIVARIATE_START)) &
    (raw_df["date"] <= pd.Timestamp(LONG_UNIVARIATE_END))
][["date", TARGET_COL]].copy()

model_df[TARGET_COL] = pd.to_numeric(model_df[TARGET_COL], errors="coerce")
model_df = model_df.dropna(subset=[TARGET_COL]).sort_values("date").reset_index(drop=True)

# Deterministic features used for professor-style trend/seasonality benchmarks.
model_df["trend"] = np.arange(1, len(model_df) + 1)
model_df["trend_sq"] = model_df["trend"] ** 2
model_df["month"] = model_df["date"].dt.month.astype("category")
model_df["log_gold_price"] = np.log(model_df[TARGET_COL])

train_df = model_df[(model_df["date"] >= TRAIN_START) & (model_df["date"] <= TRAIN_END)].copy()
valid_df = model_df[(model_df["date"] >= VALIDATION_START) & (model_df["date"] <= VALIDATION_END)].copy()
test_df  = model_df[(model_df["date"] >= TEST_START) & (model_df["date"] <= TEST_END)].copy()

if len(train_df) == 0 or len(valid_df) == 0 or len(test_df) == 0:
    raise ValueError("Train/validation/test split produced an empty section. Check input dates.")

print("✅ Dataset A loaded")
print("Rows:", len(model_df))
print("Date range:", model_df["date"].min().date(), "to", model_df["date"].max().date())
print("Train rows:", len(train_df), "|", train_df["date"].min().date(), "to", train_df["date"].max().date())
print("Validation rows:", len(valid_df), "|", valid_df["date"].min().date(), "to", valid_df["date"].max().date())
print("Test rows:", len(test_df), "|", test_df["date"].min().date(), "to", test_df["date"].max().date())
print("✅ Forecast evaluation modes: static multi-step and one-step rolling")


In [ ]:
# ======================================================================================
# CELL 4 — MAIN ARIMA / TREND-SEASONALITY / ROLLING MODELING LOGIC
# ======================================================================================
# Purpose:
# - Mirror professor style: plot series, fit trend/seasonality benchmarks, inspect ACF/PACF,
#   then evaluate ARIMA-family models separately.
# - Fix the previous KeyError by using a consistent candidate structure.
# - Prevent plain trend benchmarks from being selected as the ARIMA result.
# ======================================================================================

# --------------------------------------------------------------------------------------
# Helper functions
# --------------------------------------------------------------------------------------

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100)

def calculate_metrics(actual, predicted):
    actual = pd.Series(actual).astype(float).reset_index(drop=True)
    predicted = pd.Series(predicted).astype(float).reset_index(drop=True)
    mask = actual.notna() & predicted.notna() & np.isfinite(actual) & np.isfinite(predicted)
    actual = actual[mask]
    predicted = predicted[mask]
    residual = actual - predicted

    if len(actual) == 0:
        return {
            "n": 0, "mae": None, "mse": None, "rmse": None, "mape": None,
            "smape": None, "mean_error": None, "max_abs_error": None
        }

    return {
        "n": int(len(actual)),
        "mae": float(np.mean(np.abs(residual))),
        "mse": float(np.mean(residual ** 2)),
        "rmse": float(np.sqrt(np.mean(residual ** 2))),
        "mape": float(np.mean(np.abs(residual / actual.replace(0, np.nan))) * 100),
        "smape": smape(actual, predicted),
        "mean_error": float(np.mean(residual)),
        "max_abs_error": float(np.max(np.abs(residual))),
    }

def make_path_df(date_series, actual_series, prediction_series, model_name, evaluation_period, forecast_mode):
    path = pd.DataFrame({
        "date": pd.to_datetime(date_series).dt.strftime("%Y-%m-%d"),
        "actual": pd.Series(actual_series).astype(float).values,
        "prediction": pd.Series(prediction_series).astype(float).values,
    })
    path["residual"] = path["actual"] - path["prediction"]
    path["abs_error"] = path["residual"].abs()
    path["model_name"] = model_name
    path["evaluation_period"] = evaluation_period
    path["forecast_mode"] = forecast_mode
    return path

def order_to_string(order):
    return f"ARIMA{tuple(order)}"

def safe_aic_bic(result):
    return {
        "aic": float(result.aic) if hasattr(result, "aic") and np.isfinite(result.aic) else None,
        "bic": float(result.bic) if hasattr(result, "bic") and np.isfinite(result.bic) else None,
    }

def fit_arima(series, order, trend=None):
    series = pd.Series(series).astype(float)
    model = ARIMA(
        series,
        order=order,
        trend=trend,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return model.fit()

def static_arima_forecast(train_series, forecast_steps, order, trend=None, transform=None):
    """Static multi-step forecast. Transform can be None or 'log'."""
    y_train = pd.Series(train_series).astype(float).reset_index(drop=True)
    if transform == "log":
        fit_y = np.log(y_train)
        res = fit_arima(fit_y, order=order, trend=trend)
        pred = np.exp(res.forecast(steps=forecast_steps))
    else:
        res = fit_arima(y_train, order=order, trend=trend)
        pred = res.forecast(steps=forecast_steps)
    return pd.Series(pred).reset_index(drop=True), res

def rolling_arima_forecast(history_series, actual_series, order, trend=None, transform=None, refit_every=0):
    """
    One-step rolling forecast:
    - Fit once on the initial history.
    - Forecast one step.
    - Append the actual observation.
    - Optionally refit every N observations.
    """
    history = pd.Series(history_series).astype(float).reset_index(drop=True)
    actual = pd.Series(actual_series).astype(float).reset_index(drop=True)

    if transform == "log":
        fit_history = np.log(history)
    else:
        fit_history = history.copy()

    res = fit_arima(fit_history, order=order, trend=trend)

    preds = []
    current_series = fit_history.copy()

    for i, actual_value in enumerate(actual):
        one_pred = res.forecast(steps=1)
        pred_value = float(one_pred.iloc[0] if hasattr(one_pred, "iloc") else one_pred[0])
        if transform == "log":
            pred_value = float(np.exp(pred_value))
            new_obs_for_model = float(np.log(actual_value))
        else:
            new_obs_for_model = float(actual_value)

        preds.append(pred_value)

        current_series = pd.concat([current_series, pd.Series([new_obs_for_model])], ignore_index=True)

        if refit_every and (i + 1) % refit_every == 0:
            res = fit_arima(current_series, order=order, trend=trend)
        else:
            # append updates the state cheaply without a full refit
            try:
                res = res.append([new_obs_for_model], refit=False)
            except Exception:
                # fallback to a full refit if append fails in a specific statsmodels version
                res = fit_arima(current_series, order=order, trend=trend)

    return pd.Series(preds), res

def plot_actual_pred_residual(path_df, title, max_points=None):
    p = path_df.copy()
    if max_points is not None and len(p) > max_points:
        p = p.tail(max_points).copy()

    p["date_dt"] = pd.to_datetime(p["date"])

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    axes[0].plot(p["date_dt"], p["actual"], label="Actual Gold Price")
    axes[0].plot(p["date_dt"], p["prediction"], label="Forecast / Fitted Value")
    axes[0].set_title(title)
    axes[0].set_ylabel("Gold Price")
    axes[0].legend()

    axes[1].plot(p["date_dt"], p["residual"], label="Residual = Actual - Prediction")
    axes[1].axhline(0, linewidth=1)
    axes[1].set_ylabel("Residual")
    axes[1].set_xlabel("Date")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

def plot_acf_pacf(series, title, lags=40):
    clean_series = pd.Series(series).dropna().astype(float)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    tsaplots.plot_acf(clean_series, lags=min(lags, len(clean_series)//2 - 1), ax=axes[0])
    axes[0].set_title(f"ACF — {title}")
    tsaplots.plot_pacf(clean_series, lags=min(lags, len(clean_series)//2 - 1), ax=axes[1], method="ywm")
    axes[1].set_title(f"PACF — {title}")
    plt.tight_layout()
    plt.show()

# --------------------------------------------------------------------------------------
# Professor-style Step 1 — plot raw time series and split timeline
# --------------------------------------------------------------------------------------

print("STEP 1 — Plot the long univariate gold series first")
plt.figure(figsize=(14, 5))
plt.plot(model_df["date"], model_df[TARGET_COL], label="Gold Price")
plt.axvline(pd.Timestamp(TRAIN_END), linestyle="--", label="Train End")
plt.axvline(pd.Timestamp(VALIDATION_END), linestyle="--", label="Validation End")
plt.title("Gold Price Time Series — Dataset A")
plt.xlabel("Date")
plt.ylabel("Gold Price")
plt.legend()
plt.tight_layout()
plt.show()

timeline_df = pd.DataFrame({
    "section": ["train", "validation", "test"],
    "start": [TRAIN_START, VALIDATION_START, TEST_START],
    "end": [TRAIN_END, VALIDATION_END, TEST_END],
    "rows": [len(train_df), len(valid_df), len(test_df)],
})
display(timeline_df)

# --------------------------------------------------------------------------------------
# Professor-style Step 2 — trend and seasonality benchmark models
# --------------------------------------------------------------------------------------

print("\nSTEP 2 — Professor-style trend and seasonality benchmark models")
# These are benchmark/diagnostic models. They are not allowed to be selected as the ARIMA model.
trend_specs = [
    {
        "model_key": "linear_trend",
        "display_name": "Linear Trend Benchmark",
        "formula": "gold_price ~ trend",
        "target_scale": "price",
    },
    {
        "model_key": "exponential_log_trend",
        "display_name": "Exponential / Log Trend Benchmark",
        "formula": "log_gold_price ~ trend",
        "target_scale": "log",
    },
    {
        "model_key": "quadratic_trend",
        "display_name": "Quadratic Trend Benchmark",
        "formula": "gold_price ~ trend + trend_sq",
        "target_scale": "price",
    },
    {
        "model_key": "trend_month_seasonality",
        "display_name": "Trend + Month Seasonality Benchmark",
        "formula": "gold_price ~ trend + trend_sq + C(month)",
        "target_scale": "price",
    },
    {
        "model_key": "log_trend_month_seasonality",
        "display_name": "Log Trend + Month Seasonality Benchmark",
        "formula": "log_gold_price ~ trend + trend_sq + C(month)",
        "target_scale": "log",
    },
]

benchmark_candidates = []

for spec in trend_specs:
    try:
        fit = smf.ols(spec["formula"], data=train_df).fit()

        train_pred_raw = fit.predict(train_df)
        valid_pred_raw = fit.predict(valid_df)
        test_pred_raw = fit.predict(test_df)

        if spec["target_scale"] == "log":
            pred_train = np.exp(train_pred_raw)
            pred_valid = np.exp(valid_pred_raw)
            pred_test = np.exp(test_pred_raw)
        else:
            pred_train = train_pred_raw
            pred_valid = valid_pred_raw
            pred_test = test_pred_raw

        valid_metrics = calculate_metrics(valid_df[TARGET_COL], pred_valid)
        test_metrics = calculate_metrics(test_df[TARGET_COL], pred_test)

        row = {
            "model_key": spec["model_key"],
            "display_name": spec["display_name"],
            "model_family": "deterministic_trend_benchmark",
            "formula": spec["formula"],
            "target_scale": spec["target_scale"],
            "validation_metrics": valid_metrics,
            "test_metrics": test_metrics,
            "rsquared_train": float(fit.rsquared),
            "adj_rsquared_train": float(fit.rsquared_adj),
            "fit": fit,
            "pred_train": pd.Series(pred_train).reset_index(drop=True),
            "pred_valid": pd.Series(pred_valid).reset_index(drop=True),
            "pred_test": pd.Series(pred_test).reset_index(drop=True),
        }
        benchmark_candidates.append(row)

    except Exception as e:
        print(f"⚠️ Benchmark failed: {spec['display_name']} -> {e}")

benchmark_leaderboard = pd.DataFrame([
    {
        "model_key": c["model_key"],
        "display_name": c["display_name"],
        "validation_rmse": c["validation_metrics"]["rmse"],
        "validation_mae": c["validation_metrics"]["mae"],
        "validation_mape": c["validation_metrics"]["mape"],
        "test_rmse": c["test_metrics"]["rmse"],
        "test_mae": c["test_metrics"]["mae"],
        "test_mape": c["test_metrics"]["mape"],
        "train_r2": c["rsquared_train"],
        "formula": c["formula"],
    }
    for c in benchmark_candidates
]).sort_values("validation_rmse").reset_index(drop=True)

display(benchmark_leaderboard)

# Plot the best deterministic benchmark only as a diagnostic curve.
best_benchmark = None
if benchmark_candidates and len(benchmark_leaderboard) > 0:
    best_benchmark_key = benchmark_leaderboard.iloc[0]["model_key"]
    best_benchmark = next((c for c in benchmark_candidates if c["model_key"] == best_benchmark_key), None)

if best_benchmark is not None:
    benchmark_train_path = make_path_df(
        train_df["date"], train_df[TARGET_COL], best_benchmark["pred_train"],
        best_benchmark["display_name"], "train", "fitted_values"
    )
    benchmark_valid_path = make_path_df(
        valid_df["date"], valid_df[TARGET_COL], best_benchmark["pred_valid"],
        best_benchmark["display_name"], "validation", "static_multi_step"
    )
    benchmark_plot_path = pd.concat([benchmark_train_path, benchmark_valid_path], ignore_index=True)
    plot_actual_pred_residual(
        benchmark_plot_path,
        "Professor-Style Benchmark: Training Fitted Values + Validation Forecast",
        max_points=1600
    )

# --------------------------------------------------------------------------------------
# Step 3 — stationarity and ACF/PACF diagnostics
# --------------------------------------------------------------------------------------

print("\nSTEP 3 — Stationarity and autocorrelation diagnostics")

adf_level = stattools.adfuller(train_df[TARGET_COL].astype(float), autolag="AIC")
adf_diff = stattools.adfuller(train_df[TARGET_COL].astype(float).diff().dropna(), autolag="AIC")

adf_summary = pd.DataFrame([
    {
        "series": "gold_price_level",
        "adf_statistic": adf_level[0],
        "p_value": adf_level[1],
        "lags_used": adf_level[2],
        "observations": adf_level[3],
    },
    {
        "series": "first_difference_gold_price",
        "adf_statistic": adf_diff[0],
        "p_value": adf_diff[1],
        "lags_used": adf_diff[2],
        "observations": adf_diff[3],
    },
])
display(adf_summary)

plot_acf_pacf(train_df[TARGET_COL], "Training Gold Price Level", lags=40)
plot_acf_pacf(train_df[TARGET_COL].diff().dropna(), "First-Differenced Training Gold Price", lags=40)

# Residuals from the trend + month benchmark, if available.
trend_month_candidate = next((c for c in benchmark_candidates if c["model_key"] == "trend_month_seasonality"), None)
if trend_month_candidate is not None:
    trend_month_resid = train_df[TARGET_COL].reset_index(drop=True) - trend_month_candidate["pred_train"].reset_index(drop=True)
    plot_acf_pacf(trend_month_resid, "Residuals from Trend + Month Diagnostic", lags=40)

# --------------------------------------------------------------------------------------
# Step 4 — direct ARIMA and log ARIMA candidate comparison
# --------------------------------------------------------------------------------------

print("\nSTEP 4 — Direct ARIMA and Log ARIMA candidate comparison")
arima_candidates = []

train_y = train_df[TARGET_COL].astype(float).reset_index(drop=True)
valid_y = valid_df[TARGET_COL].astype(float).reset_index(drop=True)

for order in DIRECT_ARIMA_CANDIDATE_ORDERS:
    # Static direct ARIMA
    try:
        pred_valid_static, fit_static = static_arima_forecast(
            train_y, forecast_steps=len(valid_df), order=order, trend=None, transform=None
        )
        metrics = calculate_metrics(valid_y, pred_valid_static)
        arima_candidates.append({
            "model_key": f"direct_arima_static_{order}",
            "display_name": f"Direct {order_to_string(order)} — Static Validation Forecast",
            "model_family": "arima_family",
            "candidate_type": "direct_arima_static",
            "order": order,
            "target_scale": "price",
            "forecast_mode": "static_multi_step",
            "validation_metrics": metrics,
            "fit_aic_bic": safe_aic_bic(fit_static),
        })
    except Exception as e:
        print(f"⚠️ Direct static ARIMA failed {order}: {e}")

    # Rolling direct ARIMA
    try:
        pred_valid_roll, fit_roll = rolling_arima_forecast(
            train_y, valid_y, order=order, trend=None, transform=None, refit_every=ROLLING_REFIT_EVERY
        )
        metrics = calculate_metrics(valid_y, pred_valid_roll)
        arima_candidates.append({
            "model_key": f"direct_arima_rolling_{order}",
            "display_name": f"Direct {order_to_string(order)} — One-Step Rolling Validation",
            "model_family": "arima_family",
            "candidate_type": "direct_arima_rolling",
            "order": order,
            "target_scale": "price",
            "forecast_mode": "one_step_rolling",
            "validation_metrics": metrics,
            "fit_aic_bic": safe_aic_bic(fit_roll),
        })
    except Exception as e:
        print(f"⚠️ Direct rolling ARIMA failed {order}: {e}")

    # Static log ARIMA
    try:
        pred_valid_static, fit_static = static_arima_forecast(
            train_y, forecast_steps=len(valid_df), order=order, trend=None, transform="log"
        )
        metrics = calculate_metrics(valid_y, pred_valid_static)
        arima_candidates.append({
            "model_key": f"log_arima_static_{order}",
            "display_name": f"Log {order_to_string(order)} — Static Validation Forecast",
            "model_family": "arima_family",
            "candidate_type": "log_arima_static",
            "order": order,
            "target_scale": "log_price_reconverted_to_price",
            "forecast_mode": "static_multi_step",
            "validation_metrics": metrics,
            "fit_aic_bic": safe_aic_bic(fit_static),
        })
    except Exception as e:
        print(f"⚠️ Log static ARIMA failed {order}: {e}")

    # Rolling log ARIMA
    try:
        pred_valid_roll, fit_roll = rolling_arima_forecast(
            train_y, valid_y, order=order, trend=None, transform="log", refit_every=ROLLING_REFIT_EVERY
        )
        metrics = calculate_metrics(valid_y, pred_valid_roll)
        arima_candidates.append({
            "model_key": f"log_arima_rolling_{order}",
            "display_name": f"Log {order_to_string(order)} — One-Step Rolling Validation",
            "model_family": "arima_family",
            "candidate_type": "log_arima_rolling",
            "order": order,
            "target_scale": "log_price_reconverted_to_price",
            "forecast_mode": "one_step_rolling",
            "validation_metrics": metrics,
            "fit_aic_bic": safe_aic_bic(fit_roll),
        })
    except Exception as e:
        print(f"⚠️ Log rolling ARIMA failed {order}: {e}")

# --------------------------------------------------------------------------------------
# Step 5 — professor-style trend/seasonality + ARIMA residual correction
# --------------------------------------------------------------------------------------

print("\nSTEP 5 — Hybrid trend/seasonality + ARIMA residual correction")

HYBRID_PRICE_FORMULA = "gold_price ~ trend + trend_sq + C(month)"
HYBRID_LOG_FORMULA = "log_gold_price ~ trend + trend_sq + C(month)"

def fit_hybrid_baseline(history_df, formula, target_scale):
    fit = smf.ols(formula, data=history_df).fit()
    return fit

def hybrid_static_forecast(history_df, future_df, residual_order, formula, target_scale):
    baseline_fit = fit_hybrid_baseline(history_df, formula, target_scale)
    base_history = baseline_fit.predict(history_df)
    base_future = baseline_fit.predict(future_df)

    if target_scale == "log":
        actual_history_scale = history_df["log_gold_price"].reset_index(drop=True)
        residual_history = actual_history_scale - pd.Series(base_history).reset_index(drop=True)
        residual_fit = fit_arima(residual_history, order=residual_order, trend=None)
        residual_pred = pd.Series(residual_fit.forecast(steps=len(future_df))).reset_index(drop=True)
        final_pred = np.exp(pd.Series(base_future).reset_index(drop=True) + residual_pred)
    else:
        actual_history_scale = history_df[TARGET_COL].reset_index(drop=True)
        residual_history = actual_history_scale - pd.Series(base_history).reset_index(drop=True)
        residual_fit = fit_arima(residual_history, order=residual_order, trend=None)
        residual_pred = pd.Series(residual_fit.forecast(steps=len(future_df))).reset_index(drop=True)
        final_pred = pd.Series(base_future).reset_index(drop=True) + residual_pred

    return pd.Series(final_pred).reset_index(drop=True), baseline_fit, residual_fit

def hybrid_rolling_forecast(history_df, eval_df, residual_order, formula, target_scale, refit_every=0):
    """
    Hybrid one-step rolling:
    - Deterministic baseline is fit once on the initial history.
    - Residual ARIMA is updated with actual residuals one step at a time.
    """
    baseline_fit = fit_hybrid_baseline(history_df, formula, target_scale)
    base_history = pd.Series(baseline_fit.predict(history_df)).reset_index(drop=True)
    base_eval = pd.Series(baseline_fit.predict(eval_df)).reset_index(drop=True)

    if target_scale == "log":
        history_actual_scale = history_df["log_gold_price"].reset_index(drop=True)
    else:
        history_actual_scale = history_df[TARGET_COL].reset_index(drop=True)

    residual_history = history_actual_scale - base_history
    residual_fit = fit_arima(residual_history, order=residual_order, trend=None)

    residual_model_series = residual_history.copy()
    preds = []

    for i in range(len(eval_df)):
        residual_forecast = residual_fit.forecast(steps=1)
        residual_pred = float(residual_forecast.iloc[0] if hasattr(residual_forecast, "iloc") else residual_forecast[0])

        if target_scale == "log":
            final_pred = float(np.exp(base_eval.iloc[i] + residual_pred))
            actual_scale_value = float(eval_df["log_gold_price"].iloc[i])
        else:
            final_pred = float(base_eval.iloc[i] + residual_pred)
            actual_scale_value = float(eval_df[TARGET_COL].iloc[i])

        preds.append(final_pred)

        actual_residual = actual_scale_value - float(base_eval.iloc[i])
        residual_model_series = pd.concat([residual_model_series, pd.Series([actual_residual])], ignore_index=True)

        if refit_every and (i + 1) % refit_every == 0:
            residual_fit = fit_arima(residual_model_series, order=residual_order, trend=None)
        else:
            try:
                residual_fit = residual_fit.append([actual_residual], refit=False)
            except Exception:
                residual_fit = fit_arima(residual_model_series, order=residual_order, trend=None)

    return pd.Series(preds), baseline_fit, residual_fit

for residual_order in RESIDUAL_ARIMA_CANDIDATE_ORDERS:
    # Price-scale hybrid static
    try:
        pred_valid, base_fit, resid_fit = hybrid_static_forecast(
            train_df, valid_df, residual_order, HYBRID_PRICE_FORMULA, target_scale="price"
        )
        arima_candidates.append({
            "model_key": f"hybrid_price_static_{residual_order}",
            "display_name": f"Trend + Month + Residual {order_to_string(residual_order)} — Static Validation Forecast",
            "model_family": "arima_family",
            "candidate_type": "hybrid_price_residual_arima_static",
            "order": residual_order,
            "target_scale": "price",
            "forecast_mode": "static_multi_step",
            "formula": HYBRID_PRICE_FORMULA,
            "validation_metrics": calculate_metrics(valid_y, pred_valid),
            "fit_aic_bic": safe_aic_bic(resid_fit),
        })
    except Exception as e:
        print(f"⚠️ Hybrid price static failed {residual_order}: {e}")

    # Price-scale hybrid rolling
    try:
        pred_valid, base_fit, resid_fit = hybrid_rolling_forecast(
            train_df, valid_df, residual_order, HYBRID_PRICE_FORMULA, target_scale="price", refit_every=ROLLING_REFIT_EVERY
        )
        arima_candidates.append({
            "model_key": f"hybrid_price_rolling_{residual_order}",
            "display_name": f"Trend + Month + Residual {order_to_string(residual_order)} — One-Step Rolling Validation",
            "model_family": "arima_family",
            "candidate_type": "hybrid_price_residual_arima_rolling",
            "order": residual_order,
            "target_scale": "price",
            "forecast_mode": "one_step_rolling",
            "formula": HYBRID_PRICE_FORMULA,
            "validation_metrics": calculate_metrics(valid_y, pred_valid),
            "fit_aic_bic": safe_aic_bic(resid_fit),
        })
    except Exception as e:
        print(f"⚠️ Hybrid price rolling failed {residual_order}: {e}")

    # Log-scale hybrid static
    try:
        pred_valid, base_fit, resid_fit = hybrid_static_forecast(
            train_df, valid_df, residual_order, HYBRID_LOG_FORMULA, target_scale="log"
        )
        arima_candidates.append({
            "model_key": f"hybrid_log_static_{residual_order}",
            "display_name": f"Log Trend + Month + Residual {order_to_string(residual_order)} — Static Validation Forecast",
            "model_family": "arima_family",
            "candidate_type": "hybrid_log_residual_arima_static",
            "order": residual_order,
            "target_scale": "log_price_reconverted_to_price",
            "forecast_mode": "static_multi_step",
            "formula": HYBRID_LOG_FORMULA,
            "validation_metrics": calculate_metrics(valid_y, pred_valid),
            "fit_aic_bic": safe_aic_bic(resid_fit),
        })
    except Exception as e:
        print(f"⚠️ Hybrid log static failed {residual_order}: {e}")

    # Log-scale hybrid rolling
    try:
        pred_valid, base_fit, resid_fit = hybrid_rolling_forecast(
            train_df, valid_df, residual_order, HYBRID_LOG_FORMULA, target_scale="log", refit_every=ROLLING_REFIT_EVERY
        )
        arima_candidates.append({
            "model_key": f"hybrid_log_rolling_{residual_order}",
            "display_name": f"Log Trend + Month + Residual {order_to_string(residual_order)} — One-Step Rolling Validation",
            "model_family": "arima_family",
            "candidate_type": "hybrid_log_residual_arima_rolling",
            "order": residual_order,
            "target_scale": "log_price_reconverted_to_price",
            "forecast_mode": "one_step_rolling",
            "formula": HYBRID_LOG_FORMULA,
            "validation_metrics": calculate_metrics(valid_y, pred_valid),
            "fit_aic_bic": safe_aic_bic(resid_fit),
        })
    except Exception as e:
        print(f"⚠️ Hybrid log rolling failed {residual_order}: {e}")

# Candidate leaderboard. Only ARIMA-family candidates are here.
arima_leaderboard = pd.DataFrame([
    {
        "model_key": c["model_key"],
        "display_name": c["display_name"],
        "candidate_type": c["candidate_type"],
        "forecast_mode": c["forecast_mode"],
        "order": str(tuple(c["order"])),
        "target_scale": c["target_scale"],
        "validation_rmse": c["validation_metrics"]["rmse"],
        "validation_mae": c["validation_metrics"]["mae"],
        "validation_mape": c["validation_metrics"]["mape"],
        "aic": c.get("fit_aic_bic", {}).get("aic"),
        "bic": c.get("fit_aic_bic", {}).get("bic"),
    }
    for c in arima_candidates
]).sort_values("validation_rmse").reset_index(drop=True)

if len(arima_leaderboard) == 0:
    raise RuntimeError("No ARIMA-family candidates were successfully fitted.")

display(arima_leaderboard.head(20))

selected_key = arima_leaderboard.iloc[0]["model_key"]
selected_candidate = next(c for c in arima_candidates if c["model_key"] == selected_key)

print("\n✅ Selected ARIMA-family candidate based on validation RMSE:")
print(selected_candidate["display_name"])
print("Validation RMSE:", selected_candidate["validation_metrics"]["rmse"])
print("Forecast mode:", selected_candidate["forecast_mode"])

# --------------------------------------------------------------------------------------
# Step 6 — Recreate selected validation path and selected test path
# --------------------------------------------------------------------------------------

print("\nSTEP 6 — Build selected validation/test paths without KeyError")

combined_train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)
combined_train_valid_y = combined_train_valid_df[TARGET_COL].astype(float).reset_index(drop=True)
test_y = test_df[TARGET_COL].astype(float).reset_index(drop=True)

def predict_candidate(candidate, history_df, eval_df):
    """Return prediction series for the selected candidate using its own forecast mode."""
    order = tuple(candidate["order"])
    ctype = candidate["candidate_type"]
    mode = candidate["forecast_mode"]

    history_y = history_df[TARGET_COL].astype(float).reset_index(drop=True)
    eval_y = eval_df[TARGET_COL].astype(float).reset_index(drop=True)

    if ctype == "direct_arima_static":
        pred, fit = static_arima_forecast(history_y, len(eval_df), order, transform=None)
        return pred, fit

    if ctype == "direct_arima_rolling":
        pred, fit = rolling_arima_forecast(history_y, eval_y, order, transform=None, refit_every=ROLLING_REFIT_EVERY)
        return pred, fit

    if ctype == "log_arima_static":
        pred, fit = static_arima_forecast(history_y, len(eval_df), order, transform="log")
        return pred, fit

    if ctype == "log_arima_rolling":
        pred, fit = rolling_arima_forecast(history_y, eval_y, order, transform="log", refit_every=ROLLING_REFIT_EVERY)
        return pred, fit

    if ctype == "hybrid_price_residual_arima_static":
        pred, base_fit, resid_fit = hybrid_static_forecast(history_df, eval_df, order, HYBRID_PRICE_FORMULA, target_scale="price")
        return pred, resid_fit

    if ctype == "hybrid_price_residual_arima_rolling":
        pred, base_fit, resid_fit = hybrid_rolling_forecast(history_df, eval_df, order, HYBRID_PRICE_FORMULA, target_scale="price", refit_every=ROLLING_REFIT_EVERY)
        return pred, resid_fit

    if ctype == "hybrid_log_residual_arima_static":
        pred, base_fit, resid_fit = hybrid_static_forecast(history_df, eval_df, order, HYBRID_LOG_FORMULA, target_scale="log")
        return pred, resid_fit

    if ctype == "hybrid_log_residual_arima_rolling":
        pred, base_fit, resid_fit = hybrid_rolling_forecast(history_df, eval_df, order, HYBRID_LOG_FORMULA, target_scale="log", refit_every=ROLLING_REFIT_EVERY)
        return pred, resid_fit

    raise ValueError(f"Unknown candidate type: {ctype}")

selected_valid_pred, selected_valid_fit = predict_candidate(selected_candidate, train_df, valid_df)
selected_test_pred, selected_test_fit = predict_candidate(selected_candidate, combined_train_valid_df, test_df)

selected_valid_path = make_path_df(
    valid_df["date"], valid_df[TARGET_COL], selected_valid_pred,
    selected_candidate["display_name"], "validation", selected_candidate["forecast_mode"]
)

selected_test_path = make_path_df(
    test_df["date"], test_df[TARGET_COL], selected_test_pred,
    selected_candidate["display_name"], "test", selected_candidate["forecast_mode"]
)

selected_valid_metrics = calculate_metrics(valid_df[TARGET_COL], selected_valid_pred)
selected_test_metrics = calculate_metrics(test_df[TARGET_COL], selected_test_pred)

selected_metrics_table = pd.DataFrame([
    {"period": "validation", **selected_valid_metrics},
    {"period": "test", **selected_test_metrics},
])
display(selected_metrics_table)

# Also create a same-order static diagnostic path when the selected model is rolling.
# This makes the page honest: static long-horizon behavior is different from rolling one-step evaluation.
selected_static_valid_path = None
selected_static_test_path = None
selected_static_valid_metrics = None
selected_static_test_metrics = None

static_equivalent = dict(selected_candidate)
if "rolling" in static_equivalent["candidate_type"]:
    static_equivalent["candidate_type"] = static_equivalent["candidate_type"].replace("_rolling", "_static")
    static_equivalent["forecast_mode"] = "static_multi_step"
    static_equivalent["display_name"] = static_equivalent["display_name"].replace("One-Step Rolling", "Static")
else:
    static_equivalent = selected_candidate

try:
    static_valid_pred, _ = predict_candidate(static_equivalent, train_df, valid_df)
    static_test_pred, _ = predict_candidate(static_equivalent, combined_train_valid_df, test_df)

    selected_static_valid_path = make_path_df(
        valid_df["date"], valid_df[TARGET_COL], static_valid_pred,
        static_equivalent["display_name"], "validation", "static_multi_step"
    )
    selected_static_test_path = make_path_df(
        test_df["date"], test_df[TARGET_COL], static_test_pred,
        static_equivalent["display_name"], "test", "static_multi_step"
    )

    selected_static_valid_metrics = calculate_metrics(valid_df[TARGET_COL], static_valid_pred)
    selected_static_test_metrics = calculate_metrics(test_df[TARGET_COL], static_test_pred)

    static_metrics_table = pd.DataFrame([
        {"period": "validation_static_diagnostic", **selected_static_valid_metrics},
        {"period": "test_static_diagnostic", **selected_static_test_metrics},
    ])
    display(static_metrics_table)

except Exception as e:
    print("⚠️ Could not build static diagnostic path:", e)

# --------------------------------------------------------------------------------------
# Step 7 — professor-style plots
# --------------------------------------------------------------------------------------

print("\nSTEP 7 — Professor-style ARIMA plots")

selected_full_path = pd.concat([selected_valid_path, selected_test_path], ignore_index=True)
plot_actual_pred_residual(
    selected_full_path,
    f"Selected ARIMA-Family Model — {selected_candidate['forecast_mode'].replace('_', ' ').title()}",
    max_points=None
)

# Zoomed plots like professor's short-window forecast checks.
plot_actual_pred_residual(
    selected_valid_path.head(90),
    "Validation Zoom — First 90 Weekday Rows of 2019",
    max_points=None
)
plot_actual_pred_residual(
    selected_test_path.tail(90),
    "Recent Test Zoom — Final 90 Weekday Rows",
    max_points=None
)

if selected_static_test_path is not None:
    static_full_path = pd.concat([selected_static_valid_path, selected_static_test_path], ignore_index=True)
    plot_actual_pred_residual(
        static_full_path,
        "Static Multi-Step Diagnostic for Same Selected Structure",
        max_points=None
    )

# Residual autocorrelation for selected test path.
plot_acf_pacf(selected_test_path["residual"], "Selected ARIMA Test Residuals", lags=40)

try:
    lb_test = acorr_ljungbox(selected_test_path["residual"].dropna(), lags=[10, 20], return_df=True)
    print("\nLjung-Box test on selected ARIMA test residuals:")
    display(lb_test)
except Exception as e:
    lb_test = pd.DataFrame()
    print("⚠️ Ljung-Box test failed:", e)

# Best/worst error days.
print("\nWorst absolute prediction days in the test period")
display(selected_test_path.sort_values("abs_error", ascending=False).head(10)[["date", "actual", "prediction", "residual", "abs_error"]])

print("\nBest absolute prediction days in the test period")
display(selected_test_path.sort_values("abs_error", ascending=True).head(10)[["date", "actual", "prediction", "residual", "abs_error"]])

# Store important objects for export.
print("\n✅ Cell 4 complete. ARIMA-family model selected without allowing deterministic benchmark to win.")


In [ ]:
# ======================================================================================
# CELL 5 — ARTIFACT EXPORT
# ======================================================================================
# Purpose:
# - Export Notebook 07 JSON artifacts for the ARIMA page and validation notebook.
# - Include separate benchmark and ARIMA-family leaderboards.
# - Export selected rolling/static paths for the frontend.
# ======================================================================================

def json_safe(obj):
    """Convert pandas/numpy/statsmodels objects into JSON-safe Python objects."""
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items() if k not in ["fit"]}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return float(obj)
    if isinstance(obj, float):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return obj
    if isinstance(obj, pd.DataFrame):
        return dataframe_records(obj)
    if isinstance(obj, pd.Series):
        return obj.replace({np.nan: None}).tolist()
    return obj

def dataframe_records(df):
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].dt.strftime("%Y-%m-%d")
    return out.replace({np.nan: None}).to_dict(orient="records")

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(json_safe(payload), f, indent=2, ensure_ascii=False)
    print("✅ Wrote:", path.relative_to(PROJECT_DIR))

export_timestamp_utc = datetime.now(timezone.utc).isoformat()

# Lightweight candidate tables without fitted model objects.
benchmark_leaderboard_export = benchmark_leaderboard.copy() if "benchmark_leaderboard" in globals() else pd.DataFrame()
arima_leaderboard_export = arima_leaderboard.copy()

selected_order_json = list(selected_candidate["order"]) if selected_candidate.get("order") is not None else None

arima_results = {
    "artifact_name": "arima_results",
    "notebook": "07_arima_forecasting.ipynb",
    "version": "v4_professor_style_arima_family_with_one_step_roll_forward",
    "export_timestamp_utc": export_timestamp_utc,
    "project_identity": "Gold Nexus Alpha — professor-style gold forecasting platform",
    "model_page": "ARIMA",
    "dataset": {
        "dataset_name": "Dataset A — Long Univariate",
        "target": TARGET_COL,
        "start": LONG_UNIVARIATE_START,
        "end": LONG_UNIVARIATE_END,
        "official_forecast_cutoff": OFFICIAL_FORECAST_CUTOFF_DATE,
        "wording": "weekday-clean",
        "source_file": str(INPUT_PATH.name),
    },
    "splits": {
        "train": {"start": TRAIN_START, "end": TRAIN_END, "rows": int(len(train_df))},
        "validation": {"start": VALIDATION_START, "end": VALIDATION_END, "rows": int(len(valid_df))},
        "test": {"start": TEST_START, "end": TEST_END, "rows": int(len(test_df))},
    },
    "methodology_notes": [
        "The notebook separates deterministic trend benchmarks from ARIMA-family candidates.",
        "The selected ARIMA result is chosen only from ARIMA-family candidates.",
        "One-step rolling forecasts are included because they are directly comparable with the Naive roll-forward baseline.",
        "Static multi-step forecasts are retained as long-horizon diagnostics because ARIMA paths can flatten when actual observations are not fed back into the model.",
        "Trend and month seasonality are tested through professor-style hybrid residual ARIMA candidates."
    ],
    "selected_model": {
        "model_key": selected_candidate["model_key"],
        "display_name": selected_candidate["display_name"],
        "model_family": selected_candidate["model_family"],
        "candidate_type": selected_candidate["candidate_type"],
        "forecast_mode": selected_candidate["forecast_mode"],
        "target_scale": selected_candidate["target_scale"],
        "order": selected_order_json,
        "formula": selected_candidate.get("formula"),
        "selection_rule": "Lowest validation RMSE among ARIMA-family candidates only. Deterministic trend benchmarks are diagnostic and cannot be selected as the ARIMA model.",
    },
    "metrics": {
        "selected_validation": selected_valid_metrics,
        "selected_test": selected_test_metrics,
        "static_diagnostic_validation": selected_static_valid_metrics,
        "static_diagnostic_test": selected_static_test_metrics,
    },
    "leaderboards": {
        "deterministic_trend_benchmarks": dataframe_records(benchmark_leaderboard_export),
        "arima_family_candidates": dataframe_records(arima_leaderboard_export),
    },
    "notes_for_model_comparison": {
        "primary_metric": "RMSE",
        "primary_evaluation_mode": selected_candidate["forecast_mode"],
        "comparison_warning": "Compare rolling one-step ARIMA against rolling one-step Naive. Compare static ARIMA only against other static long-horizon forecasts.",
    },
}

arima_diagnostics = {
    "artifact_name": "arima_diagnostics",
    "notebook": "07_arima_forecasting.ipynb",
    "version": "v4_professor_style_arima_family_with_one_step_roll_forward",
    "export_timestamp_utc": export_timestamp_utc,
    "adf_tests": dataframe_records(adf_summary),
    "ljung_box_selected_test_residuals": dataframe_records(lb_test) if "lb_test" in globals() and isinstance(lb_test, pd.DataFrame) else [],
    "selected_test_worst_absolute_errors": dataframe_records(selected_test_path.sort_values("abs_error", ascending=False).head(20)),
    "selected_test_best_absolute_errors": dataframe_records(selected_test_path.sort_values("abs_error", ascending=True).head(20)),
    "residual_summary": {
        "selected_validation_residual_mean": float(selected_valid_path["residual"].mean()),
        "selected_validation_residual_std": float(selected_valid_path["residual"].std()),
        "selected_test_residual_mean": float(selected_test_path["residual"].mean()),
        "selected_test_residual_std": float(selected_test_path["residual"].std()),
    },
    "diagnostic_notes": [
        "ADF, ACF, and PACF are used to justify differencing and residual autocorrelation checks.",
        "Residual ARIMA candidates are tested after deterministic trend/month structure.",
        "Ljung-Box p-values should be interpreted as residual autocorrelation diagnostics, not final model selection alone."
    ],
}

forecast_paths = {
    "artifact_name": "arima_forecast_path",
    "notebook": "07_arima_forecasting.ipynb",
    "version": "v4_professor_style_arima_family_with_one_step_roll_forward",
    "export_timestamp_utc": export_timestamp_utc,
    "selected_model": {
        "model_key": selected_candidate["model_key"],
        "display_name": selected_candidate["display_name"],
        "forecast_mode": selected_candidate["forecast_mode"],
        "order": selected_order_json,
    },
    "paths": {
        "selected_validation": dataframe_records(selected_valid_path),
        "selected_test": dataframe_records(selected_test_path),
        "selected_validation_and_test": dataframe_records(pd.concat([selected_valid_path, selected_test_path], ignore_index=True)),
        "static_diagnostic_validation": dataframe_records(selected_static_valid_path) if selected_static_valid_path is not None else [],
        "static_diagnostic_test": dataframe_records(selected_static_test_path) if selected_static_test_path is not None else [],
        "static_diagnostic_validation_and_test": dataframe_records(
            pd.concat([selected_static_valid_path, selected_static_test_path], ignore_index=True)
        ) if selected_static_valid_path is not None and selected_static_test_path is not None else [],
    },
    "chart_defaults": {
        "frontend_default_path": "selected_validation_and_test",
        "secondary_path": "static_diagnostic_validation_and_test",
        "x_field": "date",
        "actual_field": "actual",
        "prediction_field": "prediction",
        "residual_field": "residual",
    },
}

page_arima = {
    "artifact_name": "page_arima",
    "notebook": "07_arima_forecasting.ipynb",
    "version": "v4_professor_style_arima_family_with_one_step_roll_forward",
    "export_timestamp_utc": export_timestamp_utc,
    "page_title": "ARIMA Forecasting",
    "page_subtitle": "Classical univariate forecasting with professor-style trend diagnostics, ARIMA-family selection, and one-step roll-forward evaluation.",
    "model_type": "Classical univariate time-series model",
    "dataset_window": {
        "name": "Dataset A — Long Univariate",
        "target": TARGET_COL,
        "start": LONG_UNIVARIATE_START,
        "end": LONG_UNIVARIATE_END,
        "official_cutoff": OFFICIAL_FORECAST_CUTOFF_DATE,
    },
    "split_summary": {
        "train": {"start": TRAIN_START, "end": TRAIN_END, "rows": int(len(train_df))},
        "validation": {"start": VALIDATION_START, "end": VALIDATION_END, "rows": int(len(valid_df))},
        "test": {"start": TEST_START, "end": TEST_END, "rows": int(len(test_df))},
    },
    "selected_model": arima_results["selected_model"],
    "kpi_cards": [
        {"label": "Validation RMSE", "value": selected_valid_metrics["rmse"], "period": "validation", "mode": selected_candidate["forecast_mode"]},
        {"label": "Validation MAE", "value": selected_valid_metrics["mae"], "period": "validation", "mode": selected_candidate["forecast_mode"]},
        {"label": "Validation MAPE", "value": selected_valid_metrics["mape"], "period": "validation", "mode": selected_candidate["forecast_mode"]},
        {"label": "Test RMSE", "value": selected_test_metrics["rmse"], "period": "test", "mode": selected_candidate["forecast_mode"]},
        {"label": "Test MAE", "value": selected_test_metrics["mae"], "period": "test", "mode": selected_candidate["forecast_mode"]},
        {"label": "Test MAPE", "value": selected_test_metrics["mape"], "period": "test", "mode": selected_candidate["forecast_mode"]},
    ],
    "sections": [
        {
            "title": "What this model does",
            "body": [
                "ARIMA uses historical gold price behavior to forecast future gold price movements.",
                "This notebook tests direct ARIMA, log ARIMA, and professor-style trend/month plus residual ARIMA correction.",
                "The selected ARIMA model is chosen from ARIMA-family candidates only."
            ],
        },
        {
            "title": "Why rolling forecast is shown",
            "body": [
                "One-step rolling evaluation forecasts the next observation, then updates with the actual value.",
                "This is the fairest comparison against Naive roll-forward forecasts.",
                "Static multi-step forecasts are also exported because they show long-horizon behavior without daily updating."
            ],
        },
        {
            "title": "Limitations",
            "body": [
                "ARIMA is univariate and does not directly use macroeconomic factors.",
                "Long static forecasts can flatten over multi-year horizons.",
                "Trend and month seasonality are tested but only retained if validation metrics support them."
            ],
        },
    ],
    "charts": [
        {
            "chart_id": "selected_arima_actual_vs_forecast",
            "title": "Selected ARIMA Actual vs Forecast",
            "artifact": "artifacts/models/arima_forecast_path.json",
            "path_key": "selected_validation_and_test",
            "default": True,
        },
        {
            "chart_id": "selected_arima_residuals",
            "title": "Selected ARIMA Residuals",
            "artifact": "artifacts/models/arima_forecast_path.json",
            "path_key": "selected_validation_and_test",
            "default": True,
        },
        {
            "chart_id": "static_diagnostic_actual_vs_forecast",
            "title": "Static Multi-Step Diagnostic",
            "artifact": "artifacts/models/arima_forecast_path.json",
            "path_key": "static_diagnostic_validation_and_test",
            "default": False,
        },
    ],
    "source_artifacts": [
        "artifacts/models/arima_results.json",
        "artifacts/models/arima_diagnostics.json",
        "artifacts/models/arima_forecast_path.json",
    ],
}

write_json(MODELS_ARTIFACTS_DIR / "arima_results.json", arima_results)
write_json(MODELS_ARTIFACTS_DIR / "arima_diagnostics.json", arima_diagnostics)
write_json(MODELS_ARTIFACTS_DIR / "arima_forecast_path.json", forecast_paths)
write_json(PAGES_ARTIFACTS_DIR / "page_arima.json", page_arima)

print("✅ Notebook 07 v4 artifact export complete.")


In [ ]:
# ======================================================================================
# CELL 6 — GITHUB PUSH
# ======================================================================================
# Purpose:
# - Commit and push Notebook 07 outputs to GitHub.
# ======================================================================================

files_to_add = [
    "artifacts/models/arima_results.json",
    "artifacts/models/arima_diagnostics.json",
    "artifacts/models/arima_forecast_path.json",
    "artifacts/pages/page_arima.json",
]

print("📌 Git status before add:")
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

run_cmd(["git", "add"] + files_to_add, cwd=str(PROJECT_DIR))

commit_check = run_cmd(
    ["git", "status", "--porcelain"],
    cwd=str(PROJECT_DIR),
    allow_fail=True
).stdout.strip()

if commit_check:
    commit_message = "Revise Notebook 07 ARIMA one-step rolling artifacts"
    run_cmd(["git", "commit", "-m", commit_message], cwd=str(PROJECT_DIR))
    run_cmd(["git", "push", "origin", BRANCH], cwd=str(PROJECT_DIR))
    print("✅ GitHub push complete.")
else:
    print("✅ No changes to commit. Artifacts already match repository.")

print("✅ Notebook 07 v4 professor-style ARIMA revision complete.")
